# Environment Check

In [ ]:
import sys, os, platform
print('Python:', sys.version)
print('Platform:', platform.platform())
print('Working directory:', os.getcwd())

In [ ]:
import torch, shutil
print('PyTorch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
nvcc = shutil.which('nvcc')
print('nvcc:', nvcc if nvcc else 'not found')

# Clone Repo

In [ ]:
import os
REPO_URL  = 'https://github.com/jeromereddy9/Investigating-Contrastive-Regularisation-in-State-Space-Model-Based-Sequential-Recommender-Systems'
REPO_PATH = '/content/Investigating-Contrastive-Regularisation-in-State-Space-Model-Based-Sequential-Recommender-Systems'
if not os.path.exists(REPO_PATH):
    os.system(f'git clone {REPO_URL}')
os.chdir(REPO_PATH)
print('Working directory:', os.getcwd())
print('Contents:', os.listdir('.'))

# Install Dependencies

In [ ]:
import subprocess
subprocess.run(['pip', 'install', '-q', 'numpy==2.0.2', 'recbole==1.2.0'], check=True)
print('recbole installed')

In [ ]:
import torch
if torch.cuda.is_available():
    print('GPU detected — installing mamba-ssm...')
    subprocess.run(['pip', 'install', '-q', 'causal-conv1d==1.6.2.post1'], check=True)
    subprocess.run(['pip', 'install', '-q', 'mamba-ssm==2.3.1'], check=True)
    import mamba_ssm
    print('mamba-ssm version:', mamba_ssm.__version__)
else:
    print('No GPU — skipping mamba-ssm')

# Mount Drive and Copy Dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, shutil

DATASET_SOURCE = '/content/drive/MyDrive/amazon_videogames'
DATASET_ROOT   = os.path.join(REPO_PATH, 'src/datasets/preprocessed')
DATASET_DEST   = os.path.join(DATASET_ROOT, 'amazon_videogames')

os.makedirs(DATASET_ROOT, exist_ok=True)

if not os.path.exists(DATASET_DEST):
    shutil.copytree(DATASET_SOURCE, DATASET_DEST)
    print('Dataset copied.')
else:
    print('Dataset already exists.')

print('Contents:', os.listdir(DATASET_DEST))

# Imports

In [ ]:
import sys
sys.path.insert(0, REPO_PATH)

import warnings, logging, traceback
warnings.filterwarnings('ignore')
logging.getLogger('recbole').setLevel(logging.ERROR)

import torch
from recbole.config import Config
from recbole.data import create_dataset, data_preparation
from recbole.trainer import Trainer
from recbole.utils import init_seed, init_logger

from src.utils import path_builder
from src.models.Baselines.GRU4Rec import GRU4Rec
from src.models.Baselines.SASRec import SASRec
from src.models.Baselines.CL4SRec import CL4SRec
from src.models.Baselines.DouRec import DuoRec
from src.models.Baselines.mamba4rec import Mamba4Rec
from src.models.Baselines.gated_mamba import SIGMA
from src.models.SSM_CL.mamba4rec_cl import Mamba4Rec_CL
from src.models.SSM_CL.SIGMA_cl import SIGMA_CL

print('All imports successful')

# Test Config — SIGMA_CL BPR DCL only

In [ ]:
TEST_DATASET      = 'amazon_videogames'
TEST_EPOCHS       = 3
TEST_BATCH        = 64

CONFIG_DIR        = path_builder('src/configs')
DATASET_CONFIG    = path_builder(CONFIG_DIR + '/dataset.yaml')
TRAINING_CONFIG   = path_builder(CONFIG_DIR + '/training.yaml')
MODELS_CONFIG_DIR = path_builder(CONFIG_DIR + '/models')

# Only testing SIGMA_CL BPR DCL
TESTS = [
    (SIGMA_CL, 'SIGMA_CL', 'sigma_cl', 'BPR'),
]
CL_LOSS_TYPES = ['dcl']

print(f'Test dataset : {TEST_DATASET}')
print(f'Test epochs  : {TEST_EPOCHS}')
print(f'Models       : SIGMA_CL BPR DCL only')

# Run Test

In [ ]:
def run_test(model_class, model_name, config_file, loss_type, cl_loss_type=None):
    label = model_name
    if loss_type:
        label += f'_{loss_type}'
    if cl_loss_type:
        label += f'_{cl_loss_type}'

    try:
        config_dict = {
            'epochs': TEST_EPOCHS,
            'train_batch_size': TEST_BATCH,
            'eval_batch_size': TEST_BATCH,
            'stopping_step': TEST_EPOCHS,
        }
        if loss_type:
            config_dict['loss_type'] = loss_type
        if cl_loss_type:
            config_dict['cl_loss_type'] = cl_loss_type
        if loss_type == 'BPR':
            config_dict['train_neg_sample_args'] = {
                'distribution': 'uniform',
                'sample_num': 1,
                'alpha': 1.0,
                'dynamic': False,
                'candidate_num': 0
            }

        config = Config(
            model=model_class,
            dataset=TEST_DATASET,
            config_file_list=[
                DATASET_CONFIG,
                TRAINING_CONFIG,
                path_builder(MODELS_CONFIG_DIR + f'/{config_file}.yaml'),
            ],
            config_dict=config_dict,
        )

        # Correct data path
        config['data_path'] = path_builder('src/datasets/preprocessed')

        init_seed(config['seed'], config['reproducibility'])
        init_logger(config)

        print(f'\nRunning: {label}')
        print(f'  Device : {config["device"]}')
        print(f'  GPU    : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

        dataset = create_dataset(config)
        train_data, valid_data, test_data = data_preparation(config, dataset)

        model = model_class(config, dataset).to(config['device'])
        trainer = Trainer(config, model)
        trainer.fit(train_data, valid_data, saved=False, show_progress=False)

        print(f'  PASS  {label}')
        return True

    except Exception as e:
        print(f'  FAIL  {label}')
        print(f'        {e}')
        traceback.print_exc()
        return False


# Run
passed, failed = [], []

for model_class, model_name, config_file, loss_type in TESTS:
    is_cl = model_class in (Mamba4Rec_CL, SIGMA_CL)
    if is_cl:
        for cl_loss_type in CL_LOSS_TYPES:
            ok = run_test(model_class, model_name, config_file, loss_type, cl_loss_type)
            label = f'{model_name}_{loss_type}_{cl_loss_type}'
            (passed if ok else failed).append(label)
    else:
        ok = run_test(model_class, model_name, config_file, loss_type)
        label = f'{model_name}_{loss_type}' if loss_type else model_name
        (passed if ok else failed).append(label)

print(f'\n{"="*40}')
print(f'Passed : {len(passed)}/{len(passed)+len(failed)}')
print(f'Failed : {len(failed)}')
if failed:
    print('Failed tests:')
    for f in failed:
        print(f'  - {f}')